# Three lenses: does oversampling really help churn prediction?

This notebook runs the complete experiment of the paper on Kaggle (CPU) and packs the raw results into
**one zip file**. It writes the package in `src/` of the repository to disk and runs it unchanged.

## Steps on Kaggle

1. **Settings → Accelerator: None (CPU).** No GPU is used.
2. **Add Input → Datasets** and add these four datasets (type the owner name in the search box):
   - `blastchar/telco-customer-churn`
   - `shrutimechlearn/churn-modelling`
   - `sakshigoyal7/credit-card-customers`
   - `becksddf/churn-in-telecoms-dataset`
3. **Iranian Churn and Online Retail II** (both from UCI): either set **Settings → Internet: On**
   (the notebook downloads them from UCI) or add a Kaggle copy of each. Files are recognised by their
   **column names**, so file and folder names do not matter.
4. **Quick test (recommended):** set `PART = "ALL"` and `QUICK = True` and click **Run All** (10–20 min).
   Then set `QUICK = False`.
5. Choose **PART** in the settings cell. A Kaggle session is limited to 12 hours, so the experiment is split:

   | PART | Content | Approximate time (4 CPU cores) |
   |---|---|---|
   | `"A"` | IBM Telco + Bank churn | ~2.2 h (paper run) |
   | `"B"` | Credit card + Iranian + Telecom BigML | ~3.6 h (paper run) |
   | `"C"` | Online Retail II (temporal lens; natural, 20% and 10% churn) | ~2.5 h (paper runs) |

6. **Save Version → Save & Run All (Commit)**, then download `results_churn3lensa_<PART>.zip` from the **Output** tab.

**If a session times out:** results are checkpointed per block. Create a new version and add the previous
version via **Add Input → Notebook Output**; finished blocks are skipped automatically.

In [ ]:
# ================= SETTINGS =================
PART = "A"            # "A", "B", "C" or "ALL"
QUICK = False         # True = quick test (3 folds, 2 models). Final results: False
HOURS_LIMIT = 11.3    # stop starting new blocks after this many hours (Kaggle limit: 12 h)
N_JOBS = -1           # -1 = all cores
TEMPORAL_LEVELS = ["natural", 0.20, 0.10]   # churn levels of the temporal lens (PART "C")
# ============================================
import os, sys, time, json, shutil, platform, subprocess, glob, zipfile, warnings
# Hide cosmetic warnings (e.g. LightGBM "X does not have valid feature names").
# This does not change any computation; it also applies to the parallel workers.
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
T_START = time.time()
DEADLINE = T_START + HOURS_LIMIT * 3600
WORK = os.environ.get("KAGGLE_WORK", "/kaggle/working")
INPUT_ROOT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
PKG = os.path.join(WORK, "churn3lensa")
os.makedirs(os.path.join(PKG, "src"), exist_ok=True)
os.makedirs(os.path.join(PKG, "data", "raw"), exist_ok=True)
os.makedirs(os.path.join(PKG, "results"), exist_ok=True)
print("PART:", PART, "| QUICK:", QUICK, "| working folder:", PKG)

In [ ]:
# Make sure the libraries are available (usually pre-installed on Kaggle)
need = {"imblearn": "imbalanced-learn", "xgboost": "xgboost", "lightgbm": "lightgbm", "openpyxl": "openpyxl"}
for mod, pipname in need.items():
    try:
        __import__(mod)
    except ImportError:
        print("installing", pipname, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pipname], check=False)
import numpy, pandas, scipy, sklearn, matplotlib, joblib
versions = {"python": platform.python_version(), "numpy": numpy.__version__, "pandas": pandas.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}
for mod in ("imblearn", "xgboost", "lightgbm"):
    try:
        versions[mod] = __import__(mod).__version__
    except ImportError:
        versions[mod] = "NOT INSTALLED"
versions["cpu_count"] = os.cpu_count()
versions["platform"] = platform.platform()
print(json.dumps(versions, indent=1))

## Experiment code
The following cells write the package `src/` of the repository to disk, unchanged. No need to edit them.

In [ ]:
def write_src(name, text):
    with open(os.path.join(PKG, "src", name), "w", encoding="utf-8") as f:
        f.write(text)

In [ ]:
write_src('__init__.py', r'''''')

In [ ]:
write_src('config.py', r'''"""Configuration of the three-lens churn experiment.

Every design choice that affects the results is collected here so that it can
be reported verbatim in the Methods section and changed for sensitivity analyses.
"""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
SEED = 2026

# ---------------------------------------------------------------------------
# Static datasets (Lens A = statistical metrics, Lens B = profit metrics).
# 'file' may be a glob pattern. Column names are matched case-insensitively and
# repeated spaces are collapsed, so small variations in column names still work.
# ---------------------------------------------------------------------------
STATIC_DATASETS = {
    "telco": {
        "file": "WA_Fn-UseC_-Telco-Customer-Churn*.csv",
        "source": "Kaggle: blastchar/telco-customer-churn (IBM)",
        "target": "Churn", "positive": "Yes",
        "drop": ["customerID"],
        "to_numeric": ["TotalCharges"],
    },
    "bank": {
        "file": "Churn_Modelling*.csv",
        "source": "Kaggle: shrutimechlearn/churn-modelling",
        "target": "Exited", "positive": 1,
        "drop": ["RowNumber", "CustomerId", "Surname"],
    },
    "bankchurners": {
        "file": "BankChurners*.csv",
        "source": "Kaggle: sakshigoyal7/credit-card-customers",
        "target": "Attrition_Flag", "positive": "Attrited Customer",
        "drop": ["CLIENTNUM"],
        # The last two columns of this dataset are outputs of a naive Bayes
        # classifier computed from the label -> they MUST be removed (target leakage).
        "drop_prefix": ["Naive_Bayes_Classifier"],
    },
    "iranian": {
        "file": "Customer Churn*.csv",
        "source": "UCI ML Repository id 563 (Iranian Churn)",
        "target": "Churn", "positive": 1,
        "drop": [],
    },
    "telecom_bigml": {
        "file": "bigml*.csv",
        "source": "Kaggle: becksddf/churn-in-telecoms-dataset",
        "target": "churn", "positive": True,
        "drop": ["phone number"],
    },
}

# ---------------------------------------------------------------------------
# Temporal dataset (Lens C = out-of-time validation)
# ---------------------------------------------------------------------------
RETAIL = {
    "file": "online_retail_II*",           # .xlsx from UCI or a converted .csv
    "source": "UCI ML Repository id 502 (Online Retail II)",
    "horizon_days": 90,                     # churn = no purchase in the next 90 days
    "active_window_days": 180,              # population = purchased in the previous 180 days
    "first_cutoff": "2010-06-01",
    "last_cutoff": "2011-09-01",            # last_cutoff + horizon <= end of data (2011-12-09)
    "min_train_cutoffs": 6,                 # rolling origin starts after >= 6 training snapshots
    # The 90-day churn rate of this panel is ~51% (not imbalanced). To make imbalance
    # handling meaningful, churners are subsampled WITHIN EACH snapshot to the target
    # rate; the same rows feed both the temporal and the random split.
    "ir_levels": ["natural", 0.20, 0.10],
}

# ---------------------------------------------------------------------------
# EMPC parameters (Expected Maximum Profit measure for Customer churn).
# Defaults follow Verbraken et al. (2013) / the R package 'EMP':
# CLV = 200, incentive cost d = 10, contact cost f = 1,
# probability that a targeted churner accepts the offer gamma ~ Beta(6, 14).
# ---------------------------------------------------------------------------
EMPC_PARAMS = {"clv": 200.0, "d": 10.0, "f": 1.0, "alpha": 6.0, "beta": 14.0}

# ---------------------------------------------------------------------------
# Experimental factors.
# Technique format: "name" or "name@ratio" (ratio = minority/majority after
# resampling; 1.0 = fully balanced, 0.5 = half).
# ---------------------------------------------------------------------------
TECHNIQUES_FULL = [
    "none",            # baseline without imbalance handling
    "class_weight",    # algorithm level (cost-sensitive)
    "threshold",       # algorithm level: F1-optimal threshold on a validation split
    "ros@1.0",         # random oversampling
    "rus@1.0",         # random undersampling
    "smote@1.0",
    "smote@0.5",       # non-balanced ratio (cf. Pandey et al., 2021)
    "adasyn@1.0",      # requires imbalanced-learn
    "smote_enn@1.0",   # requires imbalanced-learn
    "dirichlet@1.0",   # Dirichlet ExtSMOTE (Matharaarachchi et al., 2024)
    "easy_ensemble",   # ensemble level
    "smote_leaky@1.0", # DELIBERATELY WRONG: SMOTE before splitting -> measures leakage inflation
]
TECHNIQUES_QUICK = ["none", "class_weight", "threshold", "smote@1.0", "dirichlet@1.0",
                    "easy_ensemble", "smote_leaky@1.0"]

MODELS_FULL = ["lr", "rf", "hgb", "xgb", "lgbm"]  # xgb/lgbm are skipped if not installed
MODELS_QUICK = ["lr", "hgb"]

CV_FULL = {"repeats": 5, "folds": 5}   # 25 paired scores per condition -> Wilcoxon test
CV_QUICK = {"repeats": 1, "folds": 3}

# Moderator: the churn rate is made more extreme by subsampling churners.
# "natural" = as is. Levels above a dataset's natural rate are skipped.
IR_LEVELS_FULL = ["natural", 0.10, 0.05, 0.02]
IR_LEVELS_QUICK = ["natural"]

DIRICHLET_M = 1.0   # concentration multiplier m
K_NEIGHBORS = 5     # k for SMOTE/ADASYN/Dirichlet (as in Matharaarachchi et al.)
EASY_ENSEMBLE_N = 10
TOP_FRACTION = 0.10 # campaign capacity for the precision@10% metric
''')

In [ ]:
write_src('metrics.py', r'''"""Evaluation metrics for the three lenses.

Lens A (statistical): ROC-AUC, PR-AUC, F1, MCC, Brier score.
Lens B (profit): EMPC and the expected optimal fraction of customers targeted,
plus precision at a fixed campaign capacity.
"""
import numpy as np
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             matthews_corrcoef, brier_score_loss)


def empc(y_true, scores, clv=200.0, d=10.0, f=1.0, alpha=6.0, beta=14.0, n_gamma=200):
    """Expected Maximum Profit measure for Customer churn (Verbraken et al., 2013).

    Profit per customer when the top fraction of scores is targeted by a retention campaign:
        P(t; g) = CLV*(g*(1-delta) - phi)*pi0*F0(t) - CLV*(delta + phi)*pi1*F1(t)
    with pi0 = share of churners, F0 = TPR, F1 = FPR, delta = d/CLV, phi = f/CLV,
    and g (probability that a targeted churner accepts the offer) ~ Beta(alpha, beta).
    EMPC = E_g[ max_t P(t; g) ], computed by midpoint quadrature over Beta quantiles.
    The maximum over the empirical ROC points equals the maximum over the convex
    hull because the objective is linear in (TPR, FPR).

    Returns (empc, expected_targeted_fraction).
    """
    y = np.asarray(y_true).astype(bool)
    s = np.asarray(scores, dtype=float)
    n_pos, n_neg = y.sum(), (~y).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan, np.nan
    pi0, pi1 = n_pos / len(y), n_neg / len(y)
    order = np.argsort(-s, kind="mergesort")
    s_sorted, y_sorted = s[order], y[order]
    tp, fp = np.cumsum(y_sorted), np.cumsum(~y_sorted)
    # only keep boundaries between distinct scores (tie handling)
    last_of_group = np.r_[np.diff(s_sorted) != 0, True]
    tpr = np.r_[0.0, tp[last_of_group] / n_pos]
    fpr = np.r_[0.0, fp[last_of_group] / n_neg]
    delta, phi = d / clv, f / clv
    q = (np.arange(n_gamma) + 0.5) / n_gamma
    gammas = stats.beta.ppf(q, alpha, beta)
    profit = clv * ((gammas[:, None] * (1 - delta) - phi) * pi0 * tpr[None, :]
                    - (delta + phi) * pi1 * fpr[None, :])
    best_idx = profit.argmax(axis=1)
    frac = pi0 * tpr + pi1 * fpr
    return float(profit.max(axis=1).mean()), float(frac[best_idx].mean())


def precision_at_fraction(y_true, scores, fraction=0.10):
    y = np.asarray(y_true).astype(int)
    k = max(1, int(round(fraction * len(y))))
    top = np.argsort(-np.asarray(scores), kind="mergesort")[:k]
    return float(y[top].mean())


def best_f1_threshold(y_true, scores):
    """Threshold that maximizes F1 (used by the 'threshold' technique on validation data)."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    cand = np.unique(np.quantile(s, np.linspace(0.01, 0.99, 99)))
    f1s = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f1s))])


def evaluate(y_true, scores, threshold=0.5, empc_params=None, top_fraction=0.10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    pred = (s >= threshold).astype(int)
    out = {
        "roc_auc": roc_auc_score(y, s),
        "pr_auc": average_precision_score(y, s),
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred) if pred.min() != pred.max() else 0.0,
        "brier": brier_score_loss(y, np.clip(s, 0, 1)),
        "prec_at_top": precision_at_fraction(y, s, top_fraction),
        "threshold": threshold,
    }
    out["empc"], out["empc_frac"] = empc(y, s, **(empc_params or {}))
    return out


METRICS = ["roc_auc", "pr_auc", "f1", "mcc", "brier", "prec_at_top", "empc"]
LENS = {"roc_auc": "A", "pr_auc": "A", "f1": "A", "mcc": "A", "brier": "A",
        "prec_at_top": "B", "empc": "B"}
HIGHER_IS_BETTER = {m: (m != "brier") for m in METRICS}
''')

In [ ]:
write_src('resampling.py', r'''"""Class-imbalance handling techniques.

When imbalanced-learn is installed, ROS/RUS/SMOTE/ADASYN/SMOTE-ENN use the
official imbalanced-learn implementations (required for reported results).
Otherwise ROS/RUS/SMOTE fall back to the implementations below and
ADASYN/SMOTE-ENN are skipped. Dirichlet ExtSMOTE and EasyEnsemble always use
the implementations in this file.

All samplers are called ONLY on the training data inside a fold, after the
preprocessing has been fitted on the same training data (leakage-safe protocol).
"""
import warnings
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.neighbors import NearestNeighbors

try:
    import imblearn.over_sampling as _ios
    import imblearn.under_sampling as _ius
    import imblearn.combine as _icomb
    HAS_IMBLEARN = True
except ImportError:  # pragma: no cover
    HAS_IMBLEARN = False


class NotAvailable(Exception):
    pass


def _n_target(y, ratio):
    n_min, n_maj = int((y == 1).sum()), int((y == 0).sum())
    return n_min, n_maj, int(round(ratio * n_maj))


class RandomOver:
    def __init__(self, ratio=1.0, random_state=None):
        self.ratio, self.rng = ratio, np.random.default_rng(random_state)

    def fit_resample(self, X, y):
        n_min, _, target = _n_target(y, self.ratio)
        extra = max(0, target - n_min)
        idx = self.rng.choice(np.where(y == 1)[0], size=extra, replace=True)
        return np.vstack([X, X[idx]]), np.r_[y, y[idx]]


class RandomUnder:
    def __init__(self, ratio=1.0, random_state=None):
        self.ratio, self.rng = ratio, np.random.default_rng(random_state)

    def fit_resample(self, X, y):
        n_min = int((y == 1).sum())
        keep_maj = min(int((y == 0).sum()), int(round(n_min / self.ratio)))
        maj = self.rng.choice(np.where(y == 0)[0], size=keep_maj, replace=False)
        idx = np.r_[np.where(y == 1)[0], maj]
        return X[idx], y[idx]


class SimpleSMOTE:
    """Classical SMOTE (Chawla et al., 2002): linear interpolation towards one random neighbour (fallback)."""

    def __init__(self, ratio=1.0, k=5, random_state=None):
        self.ratio, self.k, self.rng = ratio, k, np.random.default_rng(random_state)

    def fit_resample(self, X, y):
        n_min, _, target = _n_target(y, self.ratio)
        n_new = max(0, target - n_min)
        Xm = X[y == 1]
        if n_new == 0 or len(Xm) < 2:
            return X, y
        k = min(self.k, len(Xm) - 1)
        nn = NearestNeighbors(n_neighbors=k + 1).fit(Xm)
        neigh = nn.kneighbors(Xm, return_distance=False)[:, 1:]
        base = self.rng.integers(0, len(Xm), n_new)
        pick = neigh[base, self.rng.integers(0, k, n_new)]
        gap = self.rng.random((n_new, 1))
        synth = Xm[base] + gap * (Xm[pick] - Xm[base])
        return np.vstack([X, synth]), np.r_[y, np.ones(n_new, dtype=y.dtype)]


class DirichletExtSMOTE:
    """Dirichlet ExtSMOTE, inverse-distance variant (Matharaarachchi et al., 2024).

    For every synthetic sample: pick a random minority instance r, take its k
    minority neighbours, compute the distance d_j of each neighbour to the
    coordinate-wise median of the minority class, set alpha_j = m / d_j, draw
    weights w ~ Dirichlet(alpha), and return x_new = sum_j w_j * x_j. Neighbours
    close to the centre receive larger weights, so abnormal minority instances
    have less influence.
    Implementation note: r itself is not counted as a neighbour.
    """

    def __init__(self, ratio=1.0, k=5, m=1.0, random_state=None):
        self.ratio, self.k, self.m = ratio, k, m
        self.rng = np.random.default_rng(random_state)

    def fit_resample(self, X, y):
        n_min, _, target = _n_target(y, self.ratio)
        n_new = max(0, target - n_min)
        Xm = X[y == 1]
        if n_new == 0 or len(Xm) < 2:
            return X, y
        k = min(self.k, len(Xm) - 1)
        mu = np.median(Xm, axis=0)
        dist_to_mu = np.linalg.norm(Xm - mu, axis=1)
        nn = NearestNeighbors(n_neighbors=k + 1).fit(Xm)
        neigh = nn.kneighbors(Xm, return_distance=False)[:, 1:]
        base = self.rng.integers(0, len(Xm), n_new)
        nb = neigh[base]                                   # (n_new, k)
        alpha = self.m / np.maximum(dist_to_mu[nb], 1e-8)  # (n_new, k)
        g = self.rng.gamma(alpha)                          # Dirichlet via Gamma
        w = g / np.maximum(g.sum(axis=1, keepdims=True), 1e-300)
        synth = np.einsum("nk,nkp->np", w, Xm[nb])
        return np.vstack([X, synth]), np.r_[y, np.ones(n_new, dtype=y.dtype)]


class EasyEnsemble(BaseEstimator, ClassifierMixin):
    """Average of B base models, each trained on a balanced subset (all churners
    plus an equally sized random subset of non-churners). The base model is the
    classifier under test, so that the comparison is fair across classifiers."""

    def __init__(self, base, n_estimators=10, random_state=None):
        self.base, self.n_estimators, self.random_state = base, n_estimators, random_state

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        self.models_ = []
        for _ in range(self.n_estimators):
            sub = np.r_[pos, rng.choice(neg, size=min(len(pos), len(neg)), replace=False)]
            self.models_.append(clone(self.base).fit(X[sub], y[sub]))
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_], axis=0)


def get_sampler(name, ratio=1.0, k=5, m=1.0, random_state=None):
    """Return an object with .fit_resample(X, y). Raise NotAvailable if impossible."""
    if name == "ros":
        return (_ios.RandomOverSampler(sampling_strategy=ratio, random_state=random_state)
                if HAS_IMBLEARN else RandomOver(ratio, random_state))
    if name == "rus":
        return (_ius.RandomUnderSampler(sampling_strategy=ratio, random_state=random_state)
                if HAS_IMBLEARN else RandomUnder(ratio, random_state))
    if name in ("smote", "smote_leaky"):
        return (_ios.SMOTE(sampling_strategy=ratio, k_neighbors=k, random_state=random_state)
                if HAS_IMBLEARN else SimpleSMOTE(ratio, k, random_state))
    if name == "adasyn":
        if not HAS_IMBLEARN:
            raise NotAvailable("ADASYN requires imbalanced-learn")
        return _ios.ADASYN(sampling_strategy=ratio, n_neighbors=k, random_state=random_state)
    if name == "smote_enn":
        if not HAS_IMBLEARN:
            raise NotAvailable("SMOTE-ENN requires imbalanced-learn")
        return _icomb.SMOTEENN(sampling_strategy=ratio, random_state=random_state)
    if name == "dirichlet":
        return DirichletExtSMOTE(ratio, k, m, random_state)
    raise ValueError(f"Unknown resampling technique: {name}")


def parse_technique(tech):
    """'smote@0.5' -> ('smote', 0.5); 'none' -> ('none', None)."""
    if "@" in tech:
        name, r = tech.split("@")
        return name, float(r)
    return tech, None


SAMPLER_NAMES = {"ros", "rus", "smote", "adasyn", "smote_enn", "dirichlet", "smote_leaky"}

if not HAS_IMBLEARN:
    warnings.warn("imbalanced-learn is not installed: using fallback SMOTE/ROS/RUS and "
                  "skipping ADASYN & SMOTE-ENN. Install imbalanced-learn for final results.")
''')

In [ ]:
write_src('models.py', r'''"""Classifiers and preprocessing.

Hyperparameters are fixed (not tuned) so that differences between imbalance
techniques are not confounded with tuning. The values are reported in the
Methods section; tuning can be added as a sensitivity analysis.
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:  # pragma: no cover
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:  # pragma: no cover
    HAS_LGBM = False


def make_preprocessor(X):
    """Median imputation + standardization (numeric), mode imputation + one-hot (categorical)."""
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ], sparse_threshold=0.0)


def available_models(names):
    ok = []
    for n in names:
        if n == "xgb" and not HAS_XGB:
            print("[info] xgboost not installed -> model 'xgb' skipped")
            continue
        if n == "lgbm" and not HAS_LGBM:
            print("[info] lightgbm not installed -> model 'lgbm' skipped")
            continue
        ok.append(n)
    return ok


def make_model(name, balanced=False, pos_weight=1.0, seed=0):
    cw = "balanced" if balanced else None
    if name == "lr":
        return LogisticRegression(max_iter=3000, C=1.0, class_weight=cw)
    if name == "rf":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight=cw,
                                      n_jobs=1, random_state=seed)
    if name == "hgb":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                              early_stopping=False, class_weight=cw,
                                              random_state=seed)
    if name == "xgb":
        return XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                             scale_pos_weight=pos_weight if balanced else 1.0,
                             n_jobs=1, random_state=seed, verbosity=0)
    if name == "lgbm":
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8,
                              subsample_freq=1, colsample_bytree=0.8, class_weight=cw,
                              n_jobs=1, random_state=seed, verbose=-1)
    raise ValueError(name)
''')

In [ ]:
write_src('datasets.py', r'''"""Loaders for the static datasets and the monthly snapshot builder for Online Retail II."""
import re
from pathlib import Path

import numpy as np
import pandas as pd

from . import config


def _norm(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()


def _find_file(pattern, data_dir):
    hits = sorted(Path(data_dir).glob(pattern))
    if not hits:
        raise FileNotFoundError(
            f"File '{pattern}' not found in {data_dir}. See data/README.md for download instructions.")
    return hits[0]


def _is_positive(series, positive):
    """Compare labels with the positive value tolerantly (Yes/1/True/'True')."""
    s = series.astype(str).str.strip().str.lower()
    return (s == str(positive).strip().lower()).astype(int).to_numpy()


def load_static(name, data_dir=None):
    """Return (X: feature DataFrame, y: 0/1 array, meta: dict)."""
    cfg = config.STATIC_DATASETS[name]
    path = _find_file(cfg["file"], data_dir or config.DATA_DIR)
    df = pd.read_csv(path)
    colmap = {_norm(c): c for c in df.columns}
    target = colmap.get(_norm(cfg["target"]))
    if target is None:
        raise KeyError(f"Target column '{cfg['target']}' not found in {path.name}. "
                       f"Available columns: {list(df.columns)}")
    y = _is_positive(df[target], cfg["positive"])
    drop = {target} | {colmap[_norm(c)] for c in cfg.get("drop", []) if _norm(c) in colmap}
    for pre in cfg.get("drop_prefix", []):
        drop |= {c for c in df.columns if str(c).startswith(pre)}
    X = df.drop(columns=list(drop))
    for c in cfg.get("to_numeric", []):
        if _norm(c) in colmap and colmap[_norm(c)] in X:
            X[colmap[_norm(c)]] = pd.to_numeric(X[colmap[_norm(c)]], errors="coerce")
    # text columns with very high cardinality (e.g. identifiers missed above) are dropped
    for c in list(X.columns):
        if not pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 0.5 * len(X):
            print(f"[info] {name}: column '{c}' dropped (high cardinality, probably an identifier)")
            X = X.drop(columns=c)
    X.columns = [str(c) for c in X.columns]
    meta = {"dataset": name, "file": path.name, "n": len(y), "churn_rate": float(y.mean()),
            "n_features": X.shape[1], "dropped": sorted(map(str, drop - {target}))}
    return X, y, meta


def subsample_to_rate(X, y, rate, seed):
    """Imbalance moderator: remove churners at random so that their share equals `rate`."""
    if rate == "natural" or rate >= y.mean():
        return X, y
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n_pos = int(round(rate * len(neg) / (1 - rate)))
    keep = np.r_[rng.choice(pos, size=max(n_pos, 10), replace=False), neg]
    keep.sort()
    return X.iloc[keep].reset_index(drop=True), y[keep]


# ---------------------------------------------------------------------------
# Online Retail II -> monthly customer snapshots with a forward-looking churn label
# ---------------------------------------------------------------------------
_RETAIL_ALIASES = {
    "invoice": ["invoice", "invoiceno"], "stockcode": ["stockcode"],
    "quantity": ["quantity"], "invoicedate": ["invoicedate"],
    "price": ["price", "unitprice"], "customer": ["customer id", "customerid"],
    "country": ["country"],
}


def load_retail_transactions(data_dir=None, cache=True):
    data_dir = Path(data_dir or config.DATA_DIR)
    cache_path = data_dir / "online_retail_II_clean.csv.gz"
    if cache and cache_path.exists():
        # round_trip: read floats exactly as written, so the cached and the freshly built
        # transaction tables are identical (the default parser can differ in the last digit)
        return pd.read_csv(cache_path, parse_dates=["invoicedate"], float_precision="round_trip")
    path = _find_file(config.RETAIL["file"], data_dir)
    if path.suffix.lower() in (".xlsx", ".xls"):
        print("[info] reading xlsx (1M+ rows, 2-5 minutes; the result is cached)...")
        df = pd.concat(pd.read_excel(path, sheet_name=None).values(), ignore_index=True)
    else:
        df = pd.read_csv(path, encoding_errors="replace")
    colmap = {_norm(c): c for c in df.columns}
    ren = {}
    for std, alts in _RETAIL_ALIASES.items():
        for a in alts:
            if a in colmap:
                ren[colmap[a]] = std
                break
    df = df.rename(columns=ren)[list(_RETAIL_ALIASES)]
    df = df.dropna(subset=["customer"])
    df["customer"] = df["customer"].astype(int)
    df["invoicedate"] = pd.to_datetime(df["invoicedate"])
    df["invoice"] = df["invoice"].astype(str)
    df["is_return"] = df["invoice"].str.startswith("C") | (df["quantity"] < 0)
    df["amount"] = df["quantity"] * df["price"]
    df = df.drop_duplicates()
    if cache:
        df.to_csv(cache_path, index=False, compression="gzip")
    return df


def build_snapshot(tx, cutoff, horizon_days, active_days):
    """Customer features from transactions BEFORE the cutoff; label from [cutoff, cutoff + H)."""
    cutoff = pd.Timestamp(cutoff)
    past = tx[tx["invoicedate"] < cutoff]
    buys = past[~past["is_return"]]
    last_buy = buys.groupby("customer")["invoicedate"].max()
    active = last_buy[last_buy >= cutoff - pd.Timedelta(days=active_days)].index
    if len(active) == 0:
        return pd.DataFrame()
    b = buys[buys["customer"].isin(active)]
    r = past[past["is_return"] & past["customer"].isin(active)]
    days = lambda s: (cutoff - s).dt.days

    def win(df, d, col="invoicedate"):
        return df[df[col] >= cutoff - pd.Timedelta(days=d)]

    inv = b.groupby(["customer", "invoice"]).agg(t=("invoicedate", "min"), v=("amount", "sum")).reset_index()
    g = inv.groupby("customer")
    feat = pd.DataFrame(index=pd.Index(active, name="customer"))
    feat["recency_days"] = days(g["t"].max())
    feat["tenure_days"] = days(g["t"].min())
    feat["n_invoices_total"] = g.size()
    for d in (30, 90, 365):
        w = win(inv, d, "t").groupby("customer")
        feat[f"n_invoices_{d}d"] = w.size()
        feat[f"spend_{d}d"] = w["v"].sum()
    feat["avg_basket_value"] = g["v"].mean()
    gaps = inv.sort_values("t").groupby("customer")["t"].diff().dt.days
    feat["mean_days_between"] = gaps.groupby(inv.sort_values("t")["customer"]).mean()
    feat["n_products_365d"] = win(b, 365).groupby("customer")["stockcode"].nunique()
    feat["n_returns_365d"] = win(r, 365).groupby("customer")["invoice"].nunique()
    feat["return_value_365d"] = -win(r, 365).groupby("customer")["amount"].sum()
    feat["trend_90_vs_365"] = feat["n_invoices_90d"] / (feat["n_invoices_365d"] / 4.0)
    feat["country_uk"] = (b.groupby("customer")["country"].agg(lambda s: s.mode().iat[0])
                          == "United Kingdom").astype(int)
    feat = feat.fillna(0.0)
    future = tx[(tx["invoicedate"] >= cutoff)
                & (tx["invoicedate"] < cutoff + pd.Timedelta(days=horizon_days))
                & (~tx["is_return"])]
    feat["churn"] = (~feat.index.isin(future["customer"].unique())).astype(int)
    feat["cutoff"] = cutoff
    return feat.reset_index()


def build_retail_panel(data_dir=None):
    cfg = config.RETAIL
    tx = load_retail_transactions(data_dir)
    cutoffs = pd.date_range(cfg["first_cutoff"], cfg["last_cutoff"], freq="MS")
    end = tx["invoicedate"].max()
    assert cutoffs[-1] + pd.Timedelta(days=cfg["horizon_days"]) <= end + pd.Timedelta(days=1), \
        "last_cutoff + horizon exceeds the end of the data -> last labels would be incomplete"
    panel = pd.concat([build_snapshot(tx, c, cfg["horizon_days"], cfg["active_window_days"])
                       for c in cutoffs], ignore_index=True)
    return panel
''')

In [ ]:
write_src('core.py', r'''"""One experimental unit: (training data, test data, technique, model) -> metrics.

Order of operations inside every fold (leakage-safe protocol):
 1. preprocessing is fitted ONLY on the training part and then applied to the test part
 2. resampling is applied ONLY to the preprocessed training data
 3. the model is fitted and scores the original test data (never resampled)
"""
import time
import zlib

import numpy as np
from sklearn.model_selection import train_test_split

from . import config
from .metrics import evaluate, best_f1_threshold
from .models import make_model, make_preprocessor
from .resampling import EasyEnsemble, get_sampler, parse_technique, SAMPLER_NAMES


def stable_seed(*parts):
    """Deterministic seed derived from the experimental factors (independent of run order)."""
    return zlib.crc32("|".join(map(str, parts)).encode()) % (2**31 - 1)


def fit_predict(Xtr, ytr, Xte, technique, model_name, seed):
    """Xtr/Xte are already preprocessed arrays. Returns (scores, threshold, n_train)."""
    name, ratio = parse_technique(technique)
    pos_weight = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    threshold = 0.5

    if name in SAMPLER_NAMES and name != "smote_leaky":
        sampler = get_sampler(name, ratio, config.K_NEIGHBORS, config.DIRICHLET_M, seed)
        Xtr, ytr = sampler.fit_resample(Xtr, ytr)
        model = make_model(model_name, seed=seed)
    elif name == "class_weight":
        model = make_model(model_name, balanced=True, pos_weight=pos_weight, seed=seed)
    elif name == "easy_ensemble":
        model = EasyEnsemble(make_model(model_name, seed=seed), config.EASY_ENSEMBLE_N, seed)
    elif name in ("none", "smote_leaky", "threshold"):
        model = make_model(model_name, seed=seed)
    else:
        raise ValueError(technique)

    if name == "threshold":
        # choose the F1-optimal threshold on a 20% validation split, then refit on the full fold
        Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.2, stratify=ytr, random_state=seed)
        threshold = best_f1_threshold(yv, make_model(model_name, seed=seed).fit(Xa, ya)
                                      .predict_proba(Xv)[:, 1])
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1], threshold, len(ytr)


def run_unit(Xtr_df, ytr, Xte_df, yte, technique, model_name, seed, empc_params=None):
    t0 = time.time()
    pre = make_preprocessor(Xtr_df).fit(Xtr_df)
    Xtr, Xte = pre.transform(Xtr_df), pre.transform(Xte_df)
    scores, thr, n_train = fit_predict(Xtr, ytr, Xte, technique, model_name, seed)
    res = evaluate(yte, scores, thr, empc_params or config.EMPC_PARAMS, config.TOP_FRACTION)
    res.update({"n_train_fit": n_train, "n_test": len(yte), "test_churn_rate": float(np.mean(yte)),
                "seconds": round(time.time() - t0, 3)})
    return res
''')

In [ ]:
write_src('experiment_static.py', r'''"""Static experiment: Lens A (statistical) + Lens B (profit); answers RQ1, RQ3, RQ4.

Examples:
    python -m src.experiment_static --quick
    python -m src.experiment_static --datasets telco bank --n-jobs 8
"""
import argparse
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

from . import config
from .core import run_unit, stable_seed
from .datasets import load_static, subsample_to_rate
from .metrics import evaluate
from .models import available_models, make_model, make_preprocessor
from .resampling import NotAvailable, get_sampler, parse_technique


def _task(X, y, tr, te, tech, model, seed, meta):
    try:
        res = run_unit(X.iloc[tr], y[tr], X.iloc[te], y[te], tech, model, seed)
    except NotAvailable as e:
        return {**meta, "technique": tech, "model": model, "status": f"skipped: {e}"}
    except Exception as e:  # record the error, do not stop the whole experiment
        return {**meta, "technique": tech, "model": model, "status": f"error: {type(e).__name__}: {e}"}
    return {**meta, "technique": tech, "model": model, "status": "ok", **res}


def _leaky_unit(Xres, yres, tr, te, model, seed, meta):
    """One fold of the DELIBERATELY WRONG condition (SMOTE before splitting)."""
    t0 = time.time()
    try:
        m = make_model(model, seed=seed).fit(Xres[tr], yres[tr])
        res = evaluate(yres[te], m.predict_proba(Xres[te])[:, 1], 0.5, config.EMPC_PARAMS, config.TOP_FRACTION)
    except Exception as e:
        return {**meta, "status": f"error: {type(e).__name__}: {e}"}
    res.update({"n_train_fit": len(tr), "n_test": len(te), "test_churn_rate": float(yres[te].mean()),
                "seconds": round(time.time() - t0, 3)})
    return {**meta, "status": "ok", **res}


def _leaky_jobs(X, y, tech, models, repeats, folds, meta_base):
    """DELIBERATELY WRONG: preprocessing + SMOTE on the WHOLE dataset, then CV.
    Test folds contain synthetic samples -> measures the inflation caused by leakage (RQ3)."""
    _, ratio = parse_technique(tech)
    jobs = []
    Xall = make_preprocessor(X).fit_transform(X)
    for r in range(repeats):
        seed = stable_seed(meta_base["dataset"], meta_base["ir_level"], r, tech)
        Xres, yres = get_sampler("smote_leaky", ratio, config.K_NEIGHBORS, random_state=seed).fit_resample(Xall, y)
        skf = StratifiedKFold(folds, shuffle=True, random_state=stable_seed("cv", r))
        for k, (tr, te) in enumerate(skf.split(Xres, yres)):
            for mdl in models:
                meta = {**meta_base, "repeat": r, "fold": k, "technique": tech, "model": mdl}
                jobs.append(delayed(_leaky_unit)(Xres, yres, tr, te, mdl, seed, meta))
    return jobs


def _combine(parts_dir, pattern, out_path):
    files = sorted(parts_dir.glob(pattern))
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return df


def run(datasets, techniques, models, cv, ir_levels, n_jobs, out_path, data_dir=None, deadline=None):
    """Results are checkpointed per (dataset, imbalance level) in results/parts/. Existing
    parts are skipped, so the experiment can be spread over several sessions. If `deadline`
    (epoch seconds) is given, a new block is not started when it is expected to exceed it."""
    models = available_models(models)
    parts = out_path.parent / "parts"
    parts.mkdir(parents=True, exist_ok=True)
    last_dur, stop = 0.0, False
    for ds in datasets:
        if stop:
            break
        try:
            X0, y0, meta = load_static(ds, data_dir)
        except FileNotFoundError as e:
            print(f"[skip] {e}")
            continue
        pd.DataFrame([meta]).to_csv(parts / f"meta_{ds}.csv", index=False)
        print(f"\n=== {ds}: n={meta['n']}, churn={meta['churn_rate']:.3f}, features={meta['n_features']}")
        for ir in ir_levels:
            if ir != "natural" and ir >= y0.mean():
                continue
            part = parts / f"{out_path.stem}__{ds}__{ir}.csv"
            if part.exists():
                print(f"  ir={ir}: already exists ({part.name}) -> skipped")
                continue
            if deadline and time.time() + 1.3 * last_dur > deadline:
                print(f"  [time] block {ds}/{ir} not started to stay within the session limit. Resume in the next session.")
                stop = True
                break
            X, y = subsample_to_rate(X0, y0, ir, stable_seed(ds, ir))
            base = {"dataset": ds, "ir_level": str(ir), "churn_rate": round(float(y.mean()), 4), "n": len(y)}
            jobs = []
            for r in range(cv["repeats"]):
                skf = StratifiedKFold(cv["folds"], shuffle=True, random_state=stable_seed("cv", r))
                for k, (tr, te) in enumerate(skf.split(X, y)):
                    for tech in techniques:
                        if tech.startswith("smote_leaky"):
                            continue
                        for mdl in models:
                            seed = stable_seed(ds, ir, r, k, tech, mdl)
                            jobs.append(delayed(_task)(X, y, tr, te, tech, mdl, seed,
                                                       {**base, "repeat": r, "fold": k}))
            for tech in [t for t in techniques if t.startswith("smote_leaky")]:
                jobs += _leaky_jobs(X, y, tech, models, cv["repeats"], cv["folds"], base)
            t0 = time.time()
            rows = Parallel(n_jobs=n_jobs, verbose=0)(jobs)
            pd.DataFrame(rows).to_csv(part, index=False)
            last_dur = time.time() - t0
            n_bad = sum(r.get("status") != "ok" for r in rows)
            print(f"  ir={ir}: {len(jobs)} units in {last_dur / 60:.1f} min ({n_bad} skipped/error) -> {part.name}")
    df = _combine(parts, f"{out_path.stem}__*.csv", out_path)
    metas = sorted(parts.glob("meta_*.csv"))
    if metas:
        pd.concat([pd.read_csv(f) for f in metas]).to_csv(out_path.with_name("datasets_summary.csv"), index=False)
    if len(df):
        bad = df[df["status"] != "ok"]
        print(f"\nCombined: {out_path} ({len(df)} rows; {len(bad)} skipped/error)")
        if len(bad):
            print(bad["status"].str.slice(0, 90).value_counts().head(10).to_string())
    return df


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--datasets", nargs="+", default=list(config.STATIC_DATASETS))
    p.add_argument("--techniques", nargs="+")
    p.add_argument("--models", nargs="+")
    p.add_argument("--quick", action="store_true", help="quick test mode (3 folds, 2 models)")
    p.add_argument("--n-jobs", type=int, default=-1)
    p.add_argument("--data-dir", default=None)
    p.add_argument("--out", default=None)
    a = p.parse_args()
    q = a.quick
    out = config.RESULTS_DIR / ("static_quick.csv" if q else "static_full.csv") if a.out is None else \
        __import__("pathlib").Path(a.out)
    run(a.datasets,
        a.techniques or (config.TECHNIQUES_QUICK if q else config.TECHNIQUES_FULL),
        a.models or (config.MODELS_QUICK if q else config.MODELS_FULL),
        config.CV_QUICK if q else config.CV_FULL,
        config.IR_LEVELS_QUICK if q else config.IR_LEVELS_FULL,
        a.n_jobs, out, a.data_dir)


if __name__ == "__main__":
    main()
''')

In [ ]:
write_src('experiment_temporal.py', r'''"""Temporal experiment: Lens C (out-of-time) versus a random split; answers RQ2.

Rolling-origin protocol: for every test cutoff c_test, the training data are all
snapshots with cutoff c such that c + horizon <= c_test (their labels are fully
observed before the test date). The "random" comparator uses the SAME rows but
splits them at random (stratified, identical test size) -- common practice in the
literature -- so the difference between the two measures the optimism of random splitting.

Churners are subsampled within every snapshot to the levels in config.RETAIL["ir_levels"]
(the natural 90-day churn rate of the panel is ~51%).

Example:
    python -m src.experiment_temporal --quick
"""
import argparse

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split

from . import config
from .core import run_unit, stable_seed
from .datasets import build_retail_panel
from .experiment_static import _task
from .models import available_models

FEATURE_EXCLUDE = {"customer", "churn", "cutoff"}


def subsample_panel(panel, rate):
    """Remove churners at random WITHIN EACH snapshot so that the churn rate equals `rate` (deterministic)."""
    if rate == "natural":
        return panel
    keep = []
    for c, g in panel.groupby("cutoff"):
        pos, neg = g.index[g["churn"] == 1], g.index[g["churn"] == 0]
        n_pos = min(len(pos), int(round(rate * len(neg) / (1 - rate))))
        rng = np.random.default_rng(stable_seed("retail_ir", str(pd.Timestamp(c).date()), rate))
        keep.append(neg)
        keep.append(pd.Index(rng.choice(pos, size=n_pos, replace=False)))
    idx = keep[0].append(keep[1:]).sort_values()
    return panel.loc[idx].reset_index(drop=True)


def run(techniques, models, n_jobs, out_path, data_dir=None, max_folds=None, panel=None, deadline=None,
        ir_levels=None):
    """One rolling-origin pass for every imbalance level in `ir_levels`."""
    panel0 = build_retail_panel(data_dir) if panel is None else panel
    ir_levels = ir_levels or config.RETAIL.get("ir_levels", ["natural"])
    frames = []
    for ir in ir_levels:
        print(f"\n=== temporal lens, churn level: {ir}")
        frames.append(_run_level(techniques, models, n_jobs, out_path, subsample_panel(panel0, ir), ir,
                                 max_folds, deadline))
    parts = out_path.parent / "parts"
    files = sorted(parts.glob(f"{out_path.stem}_fold*.csv")) + sorted(parts.glob(f"{out_path.stem}__*_fold*.csv"))
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"Combined temporal results: {out_path} ({len(df)} rows)")
    return df


def _run_level(techniques, models, n_jobs, out_path, panel, ir, max_folds, deadline):
    models = available_models(models)
    cfg = config.RETAIL
    suffix = "" if ir == "natural" else f"__{ir}"
    panel_path = out_path.with_name(f"retail_panel_summary{suffix}.csv")
    (panel.groupby("cutoff").agg(n=("churn", "size"), churn_rate=("churn", "mean"))
     .to_csv(panel_path))
    feats = [c for c in panel.columns if c not in FEATURE_EXCLUDE]
    cutoffs = sorted(panel["cutoff"].unique())
    H = pd.Timedelta(days=cfg["horizon_days"])
    test_cutoffs = [c for c in cutoffs
                    if sum(pd.Timestamp(x) + H <= pd.Timestamp(c) for x in cutoffs) >= cfg["min_train_cutoffs"]]
    if max_folds:
        test_cutoffs = test_cutoffs[-max_folds:]
    print(f"Panel: {len(panel)} rows, {len(cutoffs)} cutoffs; {len(test_cutoffs)} rolling-origin folds")
    import time
    parts = out_path.parent / "parts"
    parts.mkdir(parents=True, exist_ok=True)
    tag = out_path.stem + suffix  # the "natural" level keeps the original part names
    last_dur = 0.0
    for k, c_test in enumerate(test_cutoffs):
        part = parts / f"{tag}_fold{k:02d}.csv"
        if part.exists():
            print(f"  fold {k}: already exists -> skipped")
            continue
        if deadline and time.time() + 1.3 * last_dur > deadline:
            print(f"  [time] fold {k} not started to stay within the session limit. Resume in the next session.")
            break
        jobs = []
        c_test = pd.Timestamp(c_test)
        tr_mask = panel["cutoff"].map(lambda c: pd.Timestamp(c) + H <= c_test).to_numpy()
        te_mask = (panel["cutoff"] == c_test).to_numpy()
        sub = panel[tr_mask | te_mask].reset_index(drop=True)
        X, y = sub[feats], sub["churn"].to_numpy()
        tr_t, te_t = np.where(tr_mask[tr_mask | te_mask])[0], np.where(te_mask[tr_mask | te_mask])[0]
        # random comparator: same rows, same test size as the temporal fold
        tr_r, te_r = train_test_split(np.arange(len(y)), test_size=len(te_t), stratify=y,
                                      random_state=stable_seed("rand", k))
        for lens, tr, te in (("temporal", tr_t, te_t), ("random", tr_r, te_r)):
            for tech in techniques:
                if tech.startswith("smote_leaky"):
                    continue
                for mdl in models:
                    seed = stable_seed("retail", k, tech, mdl)
                    meta = {"dataset": "online_retail_ii", "lens_split": lens, "fold": k,
                            "test_cutoff": str(c_test.date()), "ir_level": str(ir)}
                    jobs.append(delayed(_task)(X, y, tr, te, tech, mdl, seed, meta))
        t0 = time.time()
        rows = Parallel(n_jobs=n_jobs)(jobs)
        pd.DataFrame(rows).to_csv(part, index=False)
        last_dur = time.time() - t0
        print(f"  fold {k} (test {c_test.date()}): {len(jobs)} units in {last_dur / 60:.1f} min")
    files = sorted(parts.glob(f"{tag}_fold*.csv"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--techniques", nargs="+")
    p.add_argument("--models", nargs="+")
    p.add_argument("--quick", action="store_true")
    p.add_argument("--n-jobs", type=int, default=-1)
    p.add_argument("--data-dir", default=None)
    p.add_argument("--out", default=None)
    a = p.parse_args()
    q = a.quick
    from pathlib import Path
    out = Path(a.out) if a.out else config.RESULTS_DIR / ("temporal_quick.csv" if q else "temporal_full.csv")
    run(a.techniques or (config.TECHNIQUES_QUICK if q else config.TECHNIQUES_FULL),
        a.models or (config.MODELS_QUICK if q else config.MODELS_FULL),
        a.n_jobs, out, a.data_dir, max_folds=3 if q else None)


if __name__ == "__main__":
    main()
''')

In [ ]:
write_src('analysis.py', r'''"""Statistical analysis, tables and exploratory figures for RQ1-RQ4.

Example:
    python -m src.analysis --static results/raw/static_full.csv --temporal results/raw/temporal_full.csv --out results/analysis
Output: <out>/tables/*.csv and <out>/figures/*.png
The exact numbers, tables and figures of the paper (natural churn rate unless stated
otherwise) are produced by src/paper.py.
"""
import argparse
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from . import config
from .metrics import METRICS, HIGHER_IS_BETTER

KEY = ["dataset", "ir_level", "model"]
PAIR = KEY + ["repeat", "fold"]
EXCLUDE_FROM_RANKING = {"smote_leaky@1.0"}

# categorical palette + text/surface tokens for the exploratory (colour) figures
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100",
     "magenta": "#e87ba4", "surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e",
     "grid": "#e4e3df", "mid": "#f0efec"}
SERIES = [C["blue"], C["orange"], C["aqua"], C["yellow"], C["magenta"]]
MARKERS = ["o", "s", "^", "D", "v"]


def holm(pvals):
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok])
    m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx])
    tmp = np.empty(m)
    tmp[idx] = np.minimum(adj, 1.0)
    out[ok] = tmp
    return out


def load(path):
    df = pd.read_csv(path)
    if "status" in df:
        df = df[df["status"] == "ok"].copy()
    df["ir_level"] = df["ir_level"].astype(str)
    return df


# ------------------------------------------------------------------ RQ1 ----
def deltas_vs_baseline(df, baseline="none"):
    """Paired differences (technique - baseline) on the same fold."""
    honest = df[~df["technique"].isin(EXCLUDE_FROM_RANKING)]
    base = honest[honest["technique"] == baseline].set_index(PAIR)[METRICS]
    rows = []
    for tech, g in honest[honest["technique"] != baseline].groupby("technique"):
        d = g.set_index(PAIR)[METRICS] - base.reindex(g.set_index(PAIR).index)
        d["technique"] = tech
        rows.append(d.reset_index())
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def wilcoxon_table(deltas):
    rows = []
    for (ds, ir, mdl, tech), g in deltas.groupby(KEY + ["technique"]):
        for m in METRICS:
            x = g[m].dropna().to_numpy()
            if not HIGHER_IS_BETTER[m]:
                x = -x
            p = stats.wilcoxon(x).pvalue if len(x) >= 5 and np.any(x != 0) else np.nan
            rows.append({"dataset": ds, "ir_level": ir, "model": mdl, "technique": tech, "metric": m,
                         "median_delta": float(np.median(x)) if len(x) else np.nan,
                         "win_rate": float(np.mean(x > 0)) if len(x) else np.nan,
                         "n_pairs": len(x), "p": p})
    t = pd.DataFrame(rows)
    t["p_holm"] = np.nan
    for _, idx in t.groupby(KEY + ["metric"]).groups.items():
        t.loc[idx, "p_holm"] = holm(t.loc[idx, "p"])
    t["verdict"] = np.select([(t["p_holm"] < 0.05) & (t["median_delta"] > 0),
                              (t["p_holm"] < 0.05) & (t["median_delta"] < 0)],
                             ["better", "worse"], "no difference")
    return t


def lens_agreement(df):
    """Kendall tau between the technique rankings produced by different metrics."""
    honest = df[~df["technique"].isin(EXCLUDE_FROM_RANKING)]
    means = honest.groupby(KEY + ["technique"])[METRICS].mean().reset_index()
    pairs = [("roc_auc", "empc"), ("pr_auc", "empc"), ("f1", "empc"), ("mcc", "empc"), ("roc_auc", "f1")]
    rows = []
    for key, g in means.groupby(KEY):
        if len(g) < 3:
            continue
        rec = dict(zip(KEY, key))
        for a, b in pairs:
            rec[f"tau_{a}_vs_{b}"] = stats.kendalltau(g[a], g[b]).statistic
        for m in ("roc_auc", "f1", "empc"):
            rec[f"best_by_{m}"] = g.loc[g[m].idxmax(), "technique"]
        rec["best_changes_f1_vs_empc"] = rec["best_by_f1"] != rec["best_by_empc"]
        rows.append(rec)
    return pd.DataFrame(rows)


def friedman_ranks(df):
    honest = df[~df["technique"].isin(EXCLUDE_FROM_RANKING)]
    means = honest.groupby(KEY + ["technique"])[METRICS].mean().reset_index()
    out = []
    for m in METRICS:
        piv = means.pivot_table(index=KEY, columns="technique", values=m).dropna()
        if piv.shape[0] < 2 or piv.shape[1] < 3:
            continue
        ranks = (-piv if HIGHER_IS_BETTER[m] else piv).rank(axis=1)
        chi, p = stats.friedmanchisquare(*[piv[c] for c in piv.columns])
        r = ranks.mean().sort_values()
        out.append(pd.DataFrame({"metric": m, "technique": r.index, "avg_rank": r.values,
                                 "friedman_chi2": chi, "friedman_p": p, "n_blocks": len(piv)}))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


# ------------------------------------------------------------------ RQ3 ----
def leakage_inflation(df):
    sub = df[df["technique"].isin(["smote@1.0", "smote_leaky@1.0"])]
    if sub["technique"].nunique() < 2:
        return pd.DataFrame()
    m = sub.groupby(KEY + ["technique"])[["roc_auc", "pr_auc", "f1", "mcc"]].mean().unstack("technique")
    out = pd.DataFrame(index=m.index)
    for met in ["roc_auc", "pr_auc", "f1", "mcc"]:
        out[f"{met}_correct"] = m[(met, "smote@1.0")]
        out[f"{met}_leaky"] = m[(met, "smote_leaky@1.0")]
        out[f"{met}_inflation"] = out[f"{met}_leaky"] - out[f"{met}_correct"]
    return out.reset_index()


# ------------------------------------------------------------------ RQ4 ----
def ir_moderator(wtab):
    t = wtab.groupby(["dataset", "ir_level", "technique", "metric"])["median_delta"].median().reset_index()
    return t.pivot_table(index=["dataset", "technique", "metric"], columns="ir_level",
                         values="median_delta").reset_index()


# ------------------------------------------------------------------ RQ2 ----
def temporal_optimism(tdf):
    if "ir_level" not in tdf:
        tdf = tdf.assign(ir_level="natural")
    tdf = tdf.assign(ir_level=tdf["ir_level"].astype(str))
    m = tdf.groupby(["lens_split", "ir_level", "technique", "model"])[METRICS].mean()
    if not {"random", "temporal"} <= set(m.index.get_level_values(0)):
        return pd.DataFrame()
    diff = (m.loc["random"] - m.loc["temporal"]).add_suffix("_optimism")
    both = m.loc["temporal"].add_suffix("_temporal").join(m.loc["random"].add_suffix("_random")).join(diff)
    return both.reset_index()


def temporal_deltas(tdf):
    out = []
    for lens, g in tdf.groupby("lens_split"):
        g = g.copy()
        g["repeat"] = 0
        w = wilcoxon_table(deltas_vs_baseline(g))
        w["lens_split"] = lens
        out.append(w)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


# ------------------------------------------------------- Exploratory figures ----
def _style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(C["surface"])
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(C["grid"])
    ax.tick_params(colors=C["ink2"], labelsize=9)
    ax.grid(axis="y", color=C["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, loc="left", fontsize=11, color=C["ink"], fontweight="bold")
    if xlabel:
        ax.set_xlabel(xlabel, color=C["ink2"], fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=C["ink2"], fontsize=9)


def figures(wtab, agree, leak, tdelta, topt, fig_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    fig_dir.mkdir(parents=True, exist_ok=True)
    if not len(wtab):  # temporal results only (no static experiment)
        wtab = pd.DataFrame(columns=["ir_level", "technique", "metric", "win_rate", "median_delta"])
    plt.rcParams.update({"figure.facecolor": C["surface"], "savefig.facecolor": C["surface"],
                         "font.size": 9})
    labels = {"roc_auc": "ROC-AUC", "pr_auc": "PR-AUC", "f1": "F1", "mcc": "MCC", "brier": "Brier",
              "prec_at_top": "Prec@10%", "empc": "EMPC"}

    # F1: win rate of each technique vs the baseline per metric (natural rate) -- RQ1
    nat = wtab[wtab["ir_level"] == "natural"]
    if len(nat):
        piv = nat.groupby(["technique", "metric"])["win_rate"].mean().unstack()[METRICS]
        cmap = LinearSegmentedColormap.from_list("div", [C["orange"], C["mid"], C["blue"]])
        fig, ax = plt.subplots(figsize=(8, 0.45 * len(piv) + 1.6))
        im = ax.imshow(piv.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(METRICS)), [labels[m] for m in METRICS])
        ax.set_yticks(range(len(piv)), piv.index)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8, color=C["ink"])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_title("How often does a technique beat the baseline 'none'?\n"
                     "(share of folds; >50% blue = wins, <50% orange = loses)",
                     loc="left", fontsize=10, color=C["ink"])
        cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cb.ax.tick_params(labelsize=8, colors=C["ink2"])
        fig.tight_layout()
        fig.savefig(fig_dir / "fig1_winrate_by_metric.png", dpi=200)
        plt.close(fig)

    # F2: agreement of rankings between lenses (Kendall tau)
    cols = [c for c in agree.columns if c.startswith("tau_")] if len(agree) else []
    if cols:
        fig, ax = plt.subplots(figsize=(7, 3.4))
        rng = np.random.default_rng(0)
        for i, c in enumerate(cols):
            v = agree[c].dropna().to_numpy()
            ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=22, color=C["blue"], alpha=0.55,
                       edgecolor=C["surface"], linewidth=0.8)
            ax.hlines(np.median(v), i - 0.25, i + 0.25, color=C["ink"], linewidth=2)
            ax.text(i, np.median(v) + 0.07, f"median {np.median(v):.2f}", ha="center", va="bottom", fontsize=8,
                    color=C["ink"], bbox=dict(boxstyle="round,pad=0.15", fc=C["surface"], ec="none", alpha=0.9))
        ax.axhline(0, color=C["ink2"], linewidth=0.8)
        ax.set_xticks(range(len(cols)), [" vs\n".join(labels[x] for x in c.replace("tau_", "").split("_vs_")) for c in cols])
        ax.set_ylim(-1.05, 1.05)
        _style(ax, "Agreement of technique rankings between metrics (Kendall τ)", None, "τ (1 = identical ranking)")
        fig.tight_layout()
        fig.savefig(fig_dir / "fig2_lens_agreement.png", dpi=200)
        plt.close(fig)

    # F3: leakage inflation
    if len(leak):
        g = leak[leak["ir_level"] == "natural"].groupby("dataset")[["roc_auc_correct", "roc_auc_leaky"]].mean()
        fig, ax = plt.subplots(figsize=(7, 0.5 * len(g) + 1.4))
        y = np.arange(len(g))
        ax.hlines(y, g["roc_auc_correct"], g["roc_auc_leaky"], color=C["grid"], linewidth=2)
        ax.scatter(g["roc_auc_correct"], y, s=64, color=C["blue"], marker="o", label="SMOTE inside the fold (correct)", zorder=3)
        ax.scatter(g["roc_auc_leaky"], y, s=64, color=C["orange"], marker="s", label="SMOTE before splitting (leaky)", zorder=3)
        for yi, (b, l) in enumerate(zip(g["roc_auc_correct"], g["roc_auc_leaky"])):
            ax.text(max(b, l) + 0.005, yi, f"+{l - b:.3f}", va="center", fontsize=8, color=C["ink2"])
        ax.set_yticks(y, g.index)
        ax.grid(axis="x", color=C["grid"])
        _style(ax, "ROC-AUC inflation caused by resampling leakage", "ROC-AUC (mean over models and folds)")
        ax.grid(axis="y", visible=False)
        ax.set_ylim(-0.6, len(g) - 0.4)
        ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.38), ncol=2)
        fig.tight_layout()
        fig.savefig(fig_dir / "fig3_leakage_inflation.png", dpi=200)
        plt.close(fig)

    # F4: imbalance moderator (small multiples, one axis per panel)
    order = ["natural", "0.1", "0.05", "0.02"]
    show = ["smote@1.0", "dirichlet@1.0", "class_weight", "easy_ensemble"]
    sub = wtab[wtab["technique"].isin(show) & wtab["ir_level"].isin(order)]
    if sub["ir_level"].nunique() > 1:
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
        for ax, met in zip(axes, ["pr_auc", "empc"]):
            s = sub[sub["metric"] == met].groupby(["technique", "ir_level"])["median_delta"].median().unstack()
            s = s[[c for c in order if c in s.columns]]
            techs = [t for t in show if t in s.index]
            for i, tech in enumerate(techs):
                ax.plot(range(s.shape[1]), s.loc[tech], color=SERIES[i], marker=MARKERS[i], linewidth=2,
                        markersize=7, label=tech)
            # end-of-line labels, nudged apart to avoid overlap
            ends = s.loc[techs].iloc[:, -1].dropna().sort_values()
            lo, hi = ax.get_ylim()
            gap, pos, last = 0.07 * (hi - lo), {}, -np.inf
            for tech, v in ends.items():
                last = max(v, last + gap)
                pos[tech] = last
            for tech, v in pos.items():
                ax.text(s.shape[1] - 1 + 0.12, v, tech, fontsize=8, color=C["ink2"], va="center")
            ax.axhline(0, color=C["ink2"], linewidth=0.8)
            ax.set_xticks(range(s.shape[1]), ["natural" if c == "natural" else f"{float(c):.0%}" for c in s.columns])
            ax.set_xlim(-0.2, s.shape[1] + 0.6)
            _style(ax, f"Δ{labels[met]} vs baseline", "churn rate", "median difference")
        axes[0].legend(frameon=False, fontsize=8, loc="best")
        fig.suptitle("Does the benefit of a technique depend on the degree of imbalance?", x=0.01, ha="left",
                     fontsize=11, fontweight="bold", color=C["ink"])
        fig.tight_layout()
        fig.savefig(fig_dir / "fig4_imbalance_moderator.png", dpi=200)
        plt.close(fig)

    # F5: optimism of random vs temporal splits
    if len(topt):
        g = topt.groupby("technique")[["roc_auc_temporal", "roc_auc_random"]].mean().sort_values("roc_auc_temporal")
        fig, ax = plt.subplots(figsize=(7, 0.45 * len(g) + 1.4))
        y = np.arange(len(g))
        ax.hlines(y, g["roc_auc_temporal"], g["roc_auc_random"], color=C["grid"], linewidth=2)
        ax.scatter(g["roc_auc_temporal"], y, s=64, color=C["blue"], marker="o", label="out-of-time (temporal)", zorder=3)
        ax.scatter(g["roc_auc_random"], y, s=64, color=C["orange"], marker="s", label="random split", zorder=3)
        ax.set_yticks(y, g.index)
        _style(ax, "Online Retail II: random split vs out-of-time", "ROC-AUC (mean over models and folds)")
        ax.grid(axis="y", visible=False)
        ax.grid(axis="x", color=C["grid"])
        ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
        fig.tight_layout()
        fig.savefig(fig_dir / "fig5_temporal_vs_random.png", dpi=200)
        plt.close(fig)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--static", default=str(config.RESULTS_DIR / "static_full.csv"))
    p.add_argument("--temporal", default=str(config.RESULTS_DIR / "temporal_full.csv"))
    p.add_argument("--out", default=str(config.RESULTS_DIR))
    a = p.parse_args()
    out = Path(a.out)
    tdir = out / "tables"
    tdir.mkdir(parents=True, exist_ok=True)
    wtab = agree = leak = tdelta = topt = pd.DataFrame()

    if Path(a.static).exists():
        df = load(a.static)
        df.groupby(KEY + ["technique"])[METRICS + ["empc_frac"]].agg(["mean", "std"]).round(4) \
            .to_csv(tdir / "T1_static_summary.csv")
        wtab = wilcoxon_table(deltas_vs_baseline(df))
        wtab.to_csv(tdir / "T2_wilcoxon_vs_baseline.csv", index=False)
        agree = lens_agreement(df)
        agree.to_csv(tdir / "T3_lens_agreement.csv", index=False)
        friedman_ranks(df).to_csv(tdir / "T3b_friedman_rank.csv", index=False)
        leak = leakage_inflation(df)
        leak.to_csv(tdir / "T4_leakage_inflation.csv", index=False)
        ir_moderator(wtab).to_csv(tdir / "T5_imbalance_moderator.csv", index=False)
        print("RQ1  share of blocks (all churn levels pooled) in which the best technique by F1 != best by EMPC:",
              f"{agree['best_changes_f1_vs_empc'].mean():.0%}" if len(agree) else "-")
        if len(leak):
            print("RQ3  mean ROC-AUC inflation caused by leakage (all churn levels pooled):", f"{leak['roc_auc_inflation'].mean():+.3f}")
    else:
        print(f"[info] {a.static} not found -> static analysis skipped")

    if Path(a.temporal).exists():
        tdf = load(a.temporal)
        topt = temporal_optimism(tdf)
        topt.to_csv(tdir / "T6_random_split_optimism.csv", index=False)
        tdelta = temporal_deltas(tdf)
        tdelta.to_csv(tdir / "T7_wilcoxon_temporal.csv", index=False)
        if len(topt):
            print("RQ2  mean ROC-AUC optimism of random vs temporal split (all churn levels pooled):",
                  f"{topt['roc_auc_optimism'].mean():+.3f}")
    else:
        print(f"[info] {a.temporal} not found -> temporal analysis skipped")

    figures(wtab, agree, leak, tdelta, topt, out / "figures")
    figures_journal(wtab, out / "figures")
    print(f"Tables: {tdir}\nFigures: {out / 'figures'}")


# journal figure sizes (inches), matching the single-column width of the IJC template
JOURNAL_SIZES = {"fig_j2_winrate": (3.4, 3.0), "fig_j3_imbalance": (3.4, 4.0)}
PRETTY = {"none": "Baseline", "class_weight": "Class weighting", "threshold": "Threshold tuning",
          "ros@1.0": "Random oversampling", "rus@1.0": "Random undersampling", "smote@1.0": "SMOTE (1.0)",
          "smote@0.5": "SMOTE (0.5)", "adasyn@1.0": "ADASYN", "smote_enn@1.0": "SMOTE-ENN",
          "dirichlet@1.0": "Dirichlet ExtSMOTE", "easy_ensemble": "EasyEnsemble"}


def figures_journal(wtab, fig_dir):
    """Grayscale figures for the International Journal of Computing (PNG 600 dpi + EPS).
    Series are distinguished by marker and line style, not colour (Figures 2 and 4 of the paper)."""
    if not len(wtab):
        return
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                         "font.size": 9, "axes.linewidth": 0.6, "figure.facecolor": "white",
                         "savefig.facecolor": "white"})
    lab = {"roc_auc": "ROC-AUC", "pr_auc": "PR-AUC", "f1": "F1", "mcc": "MCC", "brier": "Brier",
           "prec_at_top": "P@10%", "empc": "EMPC"}

    nat = wtab[wtab["ir_level"] == "natural"]
    if len(nat):
        piv = nat.groupby(["technique", "metric"])["win_rate"].mean().unstack()[METRICS]
        fig, ax = plt.subplots(figsize=JOURNAL_SIZES["fig_j2_winrate"])
        ax.imshow(piv.values, cmap="Greys", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(METRICS)), [lab[m] for m in METRICS], rotation=45, ha="right")
        ax.set_yticks(range(len(piv)), [PRETTY.get(t, t) for t in piv.index])
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{100 * v:.0f}", ha="center", va="center", fontsize=8,
                            color="white" if v > 0.55 else "black")
        ax.set_xlabel("Share of folds beating the baseline (%)")
        fig.tight_layout()
        for ext in ("png", "eps"):
            fig.savefig(fig_dir / f"fig_j2_winrate.{ext}", dpi=600)
        plt.close(fig)

    order = ["natural", "0.1", "0.05", "0.02"]
    show = ["smote@1.0", "dirichlet@1.0", "class_weight", "easy_ensemble"]
    styles = [("o", "-"), ("s", "--"), ("^", "-."), ("D", ":")]
    sub = wtab[wtab["technique"].isin(show) & wtab["ir_level"].isin(order)]
    if sub["ir_level"].nunique() > 1:
        fig, axes = plt.subplots(2, 1, figsize=JOURNAL_SIZES["fig_j3_imbalance"], sharex=True)
        for ax, met in zip(axes, ["pr_auc", "empc"]):
            s = sub[sub["metric"] == met].groupby(["technique", "ir_level"])["median_delta"].median().unstack()
            s = s[[c for c in order if c in s.columns]]
            for (mk, ls), tech in zip(styles, [t for t in show if t in s.index]):
                ax.plot(range(s.shape[1]), s.loc[tech], color="black", marker=mk, linestyle=ls,
                        linewidth=1.2, markersize=5, markerfacecolor="white", label=PRETTY.get(tech, tech))
            ax.axhline(0, color="0.5", linewidth=0.6)
            ax.set_ylabel(f"Median Δ{lab[met]}")
            ax.set_xticks(range(s.shape[1]), ["natural" if c == "natural" else f"{float(c):.0%}" for c in s.columns])
            ax.grid(axis="y", color="0.85", linewidth=0.5)
        axes[-1].set_xlabel("Churn rate after subsampling")
        h, l = axes[0].get_legend_handles_labels()
        fig.legend(h, l, fontsize=8, frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.0))
        fig.tight_layout(rect=(0, 0, 1, 0.9))
        for ext in ("png", "eps"):
            fig.savefig(fig_dir / f"fig_j3_imbalance.{ext}", dpi=600)
        plt.close(fig)


if __name__ == "__main__":
    main()
''')

In [ ]:
if PKG not in sys.path:
    sys.path.insert(0, PKG)
for m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[m]
from src import config, experiment_static, experiment_temporal, analysis
from src.datasets import load_static
print("package ready:", config.ROOT)

## Locating the datasets
Files are recognised by their **column names** and copied to `data/raw/` under standard names.

In [ ]:
import pandas as pd
RAW = str(config.DATA_DIR)
SIGNATURES = {   # dataset -> (required columns, standard file name)
    "telco":         ({"customerID", "Contract", "Churn"}, "WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    "bank":          ({"Exited", "Geography", "CreditScore"}, "Churn_Modelling.csv"),
    "bankchurners":  ({"Attrition_Flag", "CLIENTNUM"}, "BankChurners.csv"),
    "iranian":       ({"Complains", "Churn", "Seconds of Use"}, "Customer Churn.csv"),
    "telecom_bigml": ({"customer service calls", "churn"}, "bigml_telecom.csv"),
}
def norm_cols(cols):
    return {" ".join(str(c).split()) for c in cols}

found = {}
for path in glob.glob(os.path.join(INPUT_ROOT, "**", "*.csv"), recursive=True):
    if "/parts/" in path:
        continue
    try:
        cols = norm_cols(pd.read_csv(path, nrows=2, encoding_errors="replace").columns)
    except Exception:
        continue
    for ds, (sig, _) in SIGNATURES.items():
        if ds not in found and sig <= cols:
            found[ds] = path
    if "retail" not in found and {"StockCode", "Quantity"} <= cols and ("InvoiceDate" in cols):
        found["retail"] = path
if "retail" not in found:
    xl = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*.xlsx"), recursive=True)
          if "retail" in os.path.basename(p).lower()]
    if xl:
        found["retail"] = xl[0]

def try_uci(url, want):
    # Download the zip from UCI (requires Settings -> Internet: On)
    import urllib.request, io
    try:
        data = urllib.request.urlopen(url, timeout=120).read()
        zdir = os.path.join(WORK, "uci_" + want)
        zipfile.ZipFile(io.BytesIO(data)).extractall(zdir)
        for p in glob.glob(os.path.join(zdir, "**", "*"), recursive=True):
            if p.lower().endswith((".csv", ".xlsx")):
                return p
    except Exception as e:
        print(f"  UCI download failed ({type(e).__name__}: {e})")
    return None

if "iranian" not in found and PART in ("B", "ALL"):
    print("Iranian Churn not in the inputs -> trying to download it from UCI ...")
    p = try_uci("https://archive.ics.uci.edu/static/public/563/iranian+churn+dataset.zip", "iranian")
    if p: found["iranian"] = p
if "retail" not in found and PART in ("C", "ALL"):
    print("Online Retail II not in the inputs -> trying to download it from UCI (~45 MB) ...")
    p = try_uci("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", "retail")
    if p: found["retail"] = p

for ds, (sig, canon) in SIGNATURES.items():
    if ds in found:
        shutil.copy(found[ds], os.path.join(RAW, canon))
if "retail" in found:
    ext = os.path.splitext(found["retail"])[1]
    shutil.copy(found["retail"], os.path.join(RAW, "online_retail_II" + ext))
print("\nDatasets found:")
for k in list(SIGNATURES) + ["retail"]:
    print(f"  {k:14s} {'OK  ' + found[k] if k in found else 'MISSING'}")

In [ ]:
# Check the static datasets needed by this part
PART_DS = {"A": ["telco", "bank"], "B": ["bankchurners", "iranian", "telecom_bigml"], "C": [],
           "ALL": list(config.STATIC_DATASETS)}[PART]
missing = [ds for ds in PART_DS if ds not in found]
if PART in ("C", "ALL") and "retail" not in found:
    missing.append("retail")
if missing:
    raise SystemExit(f"Missing datasets: {missing}. See steps 2-3 in the first cell.")
for ds in PART_DS:
    X, y, meta = load_static(ds)
    print(f"{ds:14s} n={meta['n']:>6}  churn={meta['churn_rate']:.3f}  features={meta['n_features']}  dropped={meta['dropped']}")

In [ ]:
# Resume from a previous version (if its output was added as an input)
parts_dir = os.path.join(str(config.RESULTS_DIR), "parts")
os.makedirs(parts_dir, exist_ok=True)
n_prev = 0
for p in glob.glob(os.path.join(INPUT_ROOT, "**", "parts", "*.csv"), recursive=True):
    dst = os.path.join(parts_dir, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); n_prev += 1
print("result parts reused from previous versions:", n_prev)

## Running the experiment

In [ ]:
from pathlib import Path
q = QUICK
tag = "quick" if q else "full"
if PART_DS:
    experiment_static.run(
        PART_DS,
        config.TECHNIQUES_QUICK if q else config.TECHNIQUES_FULL,
        config.MODELS_QUICK if q else config.MODELS_FULL,
        config.CV_QUICK if q else config.CV_FULL,
        config.IR_LEVELS_QUICK if q else config.IR_LEVELS_FULL,
        N_JOBS, config.RESULTS_DIR / f"static_{tag}.csv", deadline=DEADLINE)
print(f"elapsed: {(time.time() - T_START) / 3600:.2f} h")

In [ ]:
if PART in ("C", "ALL"):
    experiment_temporal.run(
        config.TECHNIQUES_QUICK if q else config.TECHNIQUES_FULL,
        config.MODELS_QUICK if q else config.MODELS_FULL,
        N_JOBS, config.RESULTS_DIR / f"temporal_{tag}.csv",
        max_folds=3 if q else None, deadline=DEADLINE, ir_levels=TEMPORAL_LEVELS)
print(f"elapsed: {(time.time() - T_START) / 3600:.2f} h")

## Quick analysis and packing the results

In [ ]:
res = config.RESULTS_DIR
sys.argv = ["analysis", "--static", str(res / f"static_{tag}.csv"),
            "--temporal", str(res / f"temporal_{tag}.csv"), "--out", str(res / "analysis")]
try:
    analysis.main()
except Exception as e:
    print("analysis skipped:", type(e).__name__, e)

In [ ]:
versions["part"], versions["quick"] = PART, QUICK
versions["hours_elapsed"] = round((time.time() - T_START) / 3600, 2)
versions["dataset_sources"] = found
with open(res / "environment.json", "w") as f:
    json.dump(versions, f, indent=1, default=str)

summary = []
for f in sorted(glob.glob(str(res / "*.csv"))):
    try:
        d = pd.read_csv(f)
        if "status" in d:
            summary.append((os.path.basename(f), len(d), int((d["status"] != "ok").sum())))
    except Exception:
        pass
print("Result files (rows, skipped/error):")
for s in summary:
    print("  ", s)

out_zip = os.path.join(WORK, f"results_churn3lensa_{PART}{'_quick' if QUICK else ''}.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(str(res / "**" / "*"), recursive=True):
        if os.path.isfile(p):
            z.write(p, os.path.relpath(p, str(res)))
print(f"\nDONE. Download this file from the Output tab:\n  {out_zip}  ({os.path.getsize(out_zip) / 1e6:.1f} MB)")